## Meting

In [1]:
import pandas as pd
import os
import miceforest as mf
import seaborn as sns
import matplotlib.pyplot as plt


input_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\E_ResearchData\2_ResearchData\20240826'
output_data = r'T:\lab_research\RES-Folder-UPOD\ODIN-UC4\G_Output\2_Data'

In [2]:
os.chdir(input_data)

In [3]:
metingen = pd.read_parquet('meting_20240826.parquet')
metingen.studyId_0831 = metingen.studyId_0831.astype('uint64')

In [ ]:
metingen

In [ ]:
metingen.category_display_original.value_counts()

In [6]:
baseMeasurements = pd.pivot_table(metingen.loc[metingen.category_display_original=='Metingen Basisset'],
                                     columns='code_display2_original', 
                                     values='valueQuantity_value',
                                     index='studyId_0831')

Does not raise Nunique Index Exception... Nice!

In [7]:
scoreMeasurements = pd.pivot_table(metingen.loc[metingen.category_display_original == 'Scores'],
                                  columns='code_display2_original',
                                  values='valueQuantity_value',
                                  index='studyId_0831')

In [ ]:
baseMeasurements

In [ ]:
scoreMeasurements

In [ ]:
baseMeasurements.columns

In [ ]:
baseMeasurements.notna().sum()

In [ ]:
tbdropped = []
perc = 0.25
for c in baseMeasurements.columns:
    if baseMeasurements[c].notna().value_counts(True).loc[True] < perc:
        tbdropped.append(c)

baseMeasurements_filt = baseMeasurements.drop(tbdropped, axis='columns')
baseMeasurements_filt

In [ ]:
scoreMeasurements.notna().sum()

In [ ]:
tbdropped = []
perc = 0.25
for c in scoreMeasurements.columns:
    if scoreMeasurements[c].notna().value_counts(True).loc[True] < perc:
        tbdropped.append(c)

scoreMeasurements_filt = scoreMeasurements.drop(tbdropped,axis='columns')
scoreMeasurements_filt


MICE Forest Here?

---

In [15]:
ker = mf.ImputationKernel(
    baseMeasurements_filt.reset_index(),
    num_datasets=2,
    save_all_iterations_data=True,
    random_state=42)

ker.mice(iterations=20,
         n_estimators=200,
         boosting='gbdt',
         min_sum_hessian_in_leaf=50)

In [ ]:
ker.plot_imputed_distributions()

In [17]:
baseMeasurements_complete = ker.complete_data()

In [18]:
ker = mf.ImputationKernel(
    scoreMeasurements_filt.reset_index(),
    num_datasets=2,
    save_all_iterations_data=True,
    random_state=42)

ker.mice(iterations=20,
         n_estimators=200,
         boosting='gbdt',
         min_sum_hessian_in_leaf=50)

In [ ]:
ker.plot_imputed_distributions()

In [20]:
scoreMeasurements_complete = ker.complete_data()